In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [17]:
datos = pd.read_csv("ventas.csv")

In [19]:
datos.info()
datos.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_venta         500 non-null    int64  
 1   fecha_venta      499 non-null    object 
 2   id_producto      500 non-null    object 
 3   producto         500 non-null    object 
 4   categoria        450 non-null    object 
 5   region           500 non-null    object 
 6   vendedor         500 non-null    object 
 7   cantidad         500 non-null    int64  
 8   precio_unitario  446 non-null    object 
 9   descuento_pct    433 non-null    float64
 10  canal_venta      380 non-null    object 
dtypes: float64(1), int64(2), object(8)
memory usage: 43.1+ KB


,id_venta,cantidad,descuento_pct
count,500.000000,500.000000,433.000000
mean,250.500000,24.416000,14.399538
std,144.481833,15.473001,10.165742
min,1.000000,-10.000000,0.000000
25%,125.750000,12.000000,5.000000
50%,250.500000,25.000000,15.000000
75%,375.250000,37.250000,25.000000
max,500.000000,50.000000,30.000000


In [21]:
# Convert 'fecha_venta' to datetime and 'precio_unitario' to numeric
datos_limpios = datos.copy()
datos_limpios['fecha_venta'] = pd.to_datetime(datos_limpios['fecha_venta'], errors='coerce')
datos_limpios['precio_unitario'] = pd.to_numeric(datos_limpios['precio_unitario'], errors='coerce')

# Filter out rows with non-positive 'cantidad'
datos_limpios = datos_limpios[datos_limpios['cantidad'] > 0]

# Calculate 'monto'
datos_limpios['monto'] = datos_limpios['cantidad'] * datos_limpios['precio_unitario']

# Drop rows with any remaining NaN values after conversions and calculations
datos_limpios = datos_limpios.dropna()

# Create 'fecha' column
datos_limpios['fecha'] = datos_limpios['fecha_venta'].dt.date

datos_limpios.info()
datos_limpios.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 31 entries, 0 to 467
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   id_venta         31 non-null     int64         
 1   fecha_venta      31 non-null     datetime64[ns]
 2   id_producto      31 non-null     object        
 3   producto         31 non-null     object        
 4   categoria        31 non-null     object        
 5   region           31 non-null     object        
 6   vendedor         31 non-null     object        
 7   cantidad         31 non-null     int64         
 8   precio_unitario  31 non-null     float64       
 9   descuento_pct    31 non-null     float64       
 10  canal_venta      31 non-null     object        
 11  monto            31 non-null     float64       
 12  fecha            31 non-null     object        
dtypes: datetime64[ns](1), float64(3), int64(2), object(7)
memory usage: 3.4+ KB


,id_venta,fecha_venta,cantidad,precio_unitario,descuento_pct,monto
count,31.000000,31,31.000000,31.000000,31.000000,31.000000
mean,237.580645,2024-07-19 22:27:05.806451712,28.225806,2974.193548,12.903226,95683.870968
min,1.000000,2024-01-08 00:00:00,1.000000,100.000000,0.000000,500.000000
25%,146.500000,2024-04-04 00:00:00,19.000000,250.000000,5.000000,4900.000000
50%,236.000000,2024-08-03 00:00:00,30.000000,1200.000000,10.000000,38400.000000
75%,299.500000,2024-12-03 12:00:00,40.500000,5750.000000,20.000000,153000.000000
max,468.000000,2024-12-11 00:00:00,49.000000,8000.000000,30.000000,392000.000000
std,134.737714,NaN,14.474598,3250.176174,10.390235,117517.159305


In [22]:
faltantes = datos.isna().sum().reset_index()
faltantes.columns = ['columna', 'valores_faltantes']
print(faltantes)

SyntaxError: invalid syntax (1703103022.py, line 1)

In [ ]:
datos_limpios <- datos |>
  mutate(
    # La fecha viene en dos formatos: "dd/mm/aaaa" y "aaaa-mm-dd"
    fecha_venta = parse_date_time(fecha_venta, orders = c("dmy", "ymd")),

    # Estandarizar textos para evitar duplicados por mayusculas/minusculas
    producto = str_to_title(str_squish(producto)),
    categoria = replace_na(categoria, "Sin Categoria"),
    categoria = str_to_title(str_squish(categoria)),
    region = str_to_title(str_squish(region)),

    # Quitar acentos para que "Ana Lopez" y "Ana López" no salgan separados
    vendedor = iconv(vendedor, from = "", to = "ASCII//TRANSLIT"),
    vendedor = str_to_title(str_squish(vendedor)),

    canal_venta = replace_na(canal_venta, "No Especificado"),
    canal_venta = str_to_title(str_squish(canal_venta)),

    # El precio viene como texto y algunos valores tienen "$"
    precio_unitario = parse_number(as.character(precio_unitario)),

    # Si el descuento esta vacio, se toma como 0%
    descuento_pct = replace_na(descuento_pct, 0),

    # Crear variables para el analisis
    total_bruto = cantidad * precio_unitario,
    monto_descuento = total_bruto * (descuento_pct / 100),
    total_neto = total_bruto - monto_descuento,

    # Variables de tiempo
    anio = year(fecha_venta),
    mes_num = month(fecha_venta),
    mes = month(fecha_venta, label = TRUE, abbr = FALSE),
    trimestre = quarter(fecha_venta)
  ) |>
  # Para calcular ventas reales, se quitan registros sin fecha o sin precio
  drop_na(fecha_venta, precio_unitario)

In [ ]:
glimpse(datos_limpios)
summary(datos_limpios)


In [ ]:
kpi_general <- datos_limpios |>
  summarise(
    ventas_netas = sum(total_neto),
    ventas_brutas = sum(total_bruto),
    descuento_total = sum(monto_descuento),
    unidades_vendidas = sum(cantidad),
    numero_ventas = n(),
    ticket_promedio = mean(total_neto),
    precio_promedio = mean(precio_unitario)
  )

print(kpi_general)

In [ ]:
ventas_region <- datos_limpios |>
  group_by(region) |>
  summarise(
    ventas_netas = sum(total_neto),
    unidades = sum(cantidad),
    numero_ventas = n(),
    ticket_promedio = mean(total_neto),
    .groups = "drop"
  ) |>
  arrange(desc(ventas_netas))

print(ventas_region)

In [ ]:
ventas_producto <- datos_limpios |>
  group_by(producto) |>
  summarise(
    ventas_netas = sum(total_neto),
    unidades = sum(cantidad),
    numero_ventas = n(),
    .groups = "drop"
  ) |>
  arrange(desc(ventas_netas))

print(ventas_producto)

In [ ]:
ventas_categoria <- datos_limpios |>
  group_by(categoria) |>
  summarise(
    ventas_netas = sum(total_neto),
    unidades = sum(cantidad),
    numero_ventas = n(),
    .groups = "drop"
  ) |>
  arrange(desc(ventas_netas))

print(ventas_categoria)

In [ ]:
ventas_canal <- datos_limpios |>
  group_by(canal_venta) |>
  summarise(
    ventas_netas = sum(total_neto),
    unidades = sum(cantidad),
    numero_ventas = n(),
    .groups = "drop"
  ) |>
  arrange(desc(ventas_netas))

print(ventas_canal)

In [ ]:
ventas_vendedor <- datos_limpios |>
  group_by(vendedor) |>
  summarise(
    ventas_netas = sum(total_neto),
    unidades = sum(cantidad),
    numero_ventas = n(),
    ticket_promedio = mean(total_neto),
    .groups = "drop"
  ) |>
  arrange(desc(ventas_netas))

print(ventas_vendedor)

In [ ]:
ventas_mes <- datos_limpios |>
  group_by(anio, mes_num, mes) |>
  summarise(
    ventas_netas = sum(total_neto),
    unidades = sum(cantidad),
    numero_ventas = n(),
    .groups = "drop"
  ) |>
  arrange(anio, mes_num)

print(ventas_mes)

In [ ]:
pareto_producto <- ventas_producto |>
  mutate(
    porcentaje = ventas_netas / sum(ventas_netas),
    porcentaje_acumulado = cumsum(porcentaje)
  )

print(pareto_producto)

In [ ]:
grafica_producto <- ggplot(ventas_producto, aes(x = reorder(producto, ventas_netas), y = ventas_netas)) +
  geom_col() +
  coord_flip() +
  scale_y_continuous(labels = dollar_format(prefix = "$", big.mark = ",")) +
  labs(
    title = "Ventas netas por producto",
    x = "Producto",
    y = "Ventas netas (MXN)"
  ) +
  theme_minimal()

print(grafica_producto)

In [ ]:
grafica_region <- ggplot(ventas_region, aes(x = reorder(region, ventas_netas), y = ventas_netas)) +
  geom_col() +
  coord_flip() +
  scale_y_continuous(labels = dollar_format(prefix = "$", big.mark = ",")) +
  labs(
    title = "Ventas netas por region",
    x = "Region",
    y = "Ventas netas (MXN)"
  ) +
  theme_minimal()

print(grafica_region)

In [ ]:
grafica_mes <- ggplot(ventas_mes, aes(x = mes_num, y = ventas_netas)) +
  geom_line(linewidth = 1) +
  geom_point(size = 2) +
  scale_x_continuous(breaks = 1:12) +
  scale_y_continuous(labels = dollar_format(prefix = "$", big.mark = ",")) +
  labs(
    title = "Tendencia mensual de ventas netas",
    x = "Mes",
    y = "Ventas netas (MXN)"
  ) +
  theme_minimal()

print(grafica_mes)

In [ ]:
grafica_canal <- ggplot(ventas_canal, aes(x = reorder(canal_venta, ventas_netas), y = ventas_netas)) +
  geom_col() +
  coord_flip() +
  scale_y_continuous(labels = dollar_format(prefix = "$", big.mark = ",")) +
  labs(
    title = "Ventas netas por canal de venta",
    x = "Canal de venta",
    y = "Ventas netas (MXN)"
  ) +
  theme_minimal()

print(grafica_canal)

In [ ]:
grafica_pareto <- ggplot(pareto_producto, aes(x = reorder(producto, -ventas_netas))) +
  geom_col(aes(y = ventas_netas)) +
  geom_line(aes(y = porcentaje_acumulado * max(ventas_netas), group = 1), linewidth = 1) +
  geom_point(aes(y = porcentaje_acumulado * max(ventas_netas)), size = 2) +
  scale_y_continuous(
    name = "Ventas netas",
    labels = dollar_format(prefix = "$", big.mark = ","),
    sec.axis = sec_axis(~ . / max(pareto_producto$ventas_netas),
                        name = "Porcentaje acumulado",
                        labels = percent_format())
  ) +
  labs(
    title = "Analisis de Pareto por producto",
    x = "Producto"
  ) +
  theme_minimal()

print(grafica_pareto)
